In [17]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import pickle
import json
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error

df = pd.read_csv("vehicleprice.csv")

df.drop(columns=["Car/Suv"], inplace=True, errors="ignore")

df.drop(['Location', 'Title'], axis=1, inplace=True)

df.dropna(subset=['Brand', 'Model', 'Price', 'Year', 'Kilometres', 'Doors', 'Seats', 'BodyType'], inplace=True)

df['UsedOrNew'] = df['UsedOrNew'].astype(str).str.strip().str.upper()
df = df[df['UsedOrNew'] != 'DEMO']

df['Kilometres'] = pd.to_numeric(df['Kilometres'], errors='coerce')
df['Price'] = pd.to_numeric(df['Price'], errors='coerce')

df.dropna(subset=['Kilometres', 'Price'], inplace=True)

df['FuelConsumption_new'] = df['FuelConsumption'].str.extract(r'(\d+\.?\d*)').astype(float)
df['Engine_new'] = df['Engine'].str.extract(r'(\d+\.?\d*)').astype(float)
df['Cylinders_new'] = df['CylindersinEngine'].str.extract(r'(\d+)').astype(float)
df['Doors_new'] = df['Doors'].str.extract(r'(\d+)').astype(float)
df['Seats_new'] = df['Seats'].str.extract(r'(\d+)').astype(float)

df['Car_Age'] = 2025 - df['Year']
df['KM_per_Year'] = df['Kilometres'] / (df['Car_Age'] + 1)
df['log_KM'] = np.log1p(df['Kilometres'])
df['log_Engine'] = np.log1p(df['Engine_new'])
df['log_FuelCons'] = np.log1p(df['FuelConsumption_new'])

categorical_cols = ['Brand', 'Model', 'UsedOrNew', 'DriveType', 'FuelType', 'ColourExtInt', 'BodyType', 'Transmission']

for col in categorical_cols:
    df[col] = df[col].astype('category')

df.drop(['FuelConsumption', 'Engine', 'CylindersinEngine', 'Doors', 'Seats'], axis=1, inplace=True)

df.dropna(inplace=True)

cats_info = { "unique_values": {
        col: sorted(df[col].astype(str).unique().tolist())
        for col in categorical_cols
    },
    "brand_to_models": (
        df.groupby("Brand")["Model"]
        .apply(lambda x: sorted(x.astype(str).unique().tolist()))
        .to_dict()
    )
}

with open("cats_info.json", "w", encoding="utf-8") as f:
    json.dump(cats_info, f, indent=2)

encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    le.fit(df[col].astype(str))
    encoders[col] = le

with open("encoders.pkl", "wb") as f:
    pickle.dump(encoders, f)

y = np.log1p(df['Price'])
X = df.drop('Price', axis=1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

train_data = lgb.Dataset( X_train, label=y_train, categorical_feature=categorical_cols, free_raw_data=False)

val_data = lgb.Dataset( X_test, label=y_test, categorical_feature=categorical_cols, reference=train_data, free_raw_data=False)

params = { "objective": "regression", "metric": "mae", "learning_rate": 0.03, "num_leaves": 128, "verbosity": -1}

model = lgb.train( params, train_data, num_boost_round=2000, valid_sets=[val_data], callbacks=[lgb.early_stopping(100)])

with open("best_model.pkl", "wb") as f:
    pickle.dump(model, f)

preds = np.expm1(model.predict(X_test))
y_true = np.expm1(y_test)

mae = mean_absolute_error(y_true, preds)
print("MAE:", round(mae, 2))


/var/folders/4p/bhgwyy5s14jfzbktp_13q1km0000gn/T/ipykernel_13653/3740792476.py:52: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("Brand")["Model"]


Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[411]	valid_0's l1: 0.113812
MAE: 3764.4
